In [1]:
import os
import numpy as np
import pandas as pd
import librosa
from tqdm import tqdm


In [2]:
DATA_DIR = r"D:\SER_Cross\data\processed"

TRAIN_CSV = os.path.join(DATA_DIR, "train.csv")
VAL_CSV   = os.path.join(DATA_DIR, "val.csv")
TEST_CSV  = os.path.join(DATA_DIR, "test.csv")

FEATURE_DIR = r"D:\SER_Cross\features"
os.makedirs(FEATURE_DIR, exist_ok=True)

for split in ["train", "val", "test"]:
    os.makedirs(os.path.join(FEATURE_DIR, split), exist_ok=True)


In [3]:
SAMPLE_RATE = 16000
N_MELS = 128
N_FFT = 400          # 25 ms
HOP_LENGTH = 160     # 10 ms
MAX_LEN = 300        # time frames (≈ 3 seconds)


In [4]:
def extract_log_mel(wav_path):
    y, sr = librosa.load(wav_path, sr=SAMPLE_RATE, mono=True)

    mel = librosa.feature.melspectrogram(
        y=y,
        sr=sr,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        n_mels=N_MELS
    )

    log_mel = librosa.power_to_db(mel, ref=np.max)

    # Pad / truncate to fixed length
    if log_mel.shape[1] < MAX_LEN:
        pad_width = MAX_LEN - log_mel.shape[1]
        log_mel = np.pad(log_mel, ((0, 0), (0, pad_width)), mode="constant")
    else:
        log_mel = log_mel[:, :MAX_LEN]

    return log_mel


In [5]:
emotion_to_label = {
    "neutral": 0,
    "happy": 1,
    "sad": 2,
    "angry": 3,
    "fear": 4
}


In [6]:
def process_split(csv_path, split_name):
    df = pd.read_csv(csv_path)

    X, y = [], []

    for _, row in tqdm(df.iterrows(), total=len(df)):
        try:
            feature = extract_log_mel(row["path"])
            X.append(feature)
            y.append(emotion_to_label[row["emotion"]])
        except Exception as e:
            print("Error:", row["path"], e)

    X = np.array(X)
    y = np.array(y)

    np.save(os.path.join(FEATURE_DIR, split_name, "X.npy"), X)
    np.save(os.path.join(FEATURE_DIR, split_name, "y.npy"), y)

    print(f"{split_name} saved:", X.shape, y.shape)


In [7]:
process_split(TRAIN_CSV, "train")


  0%|          | 0/5068 [00:00<?, ?it/s]c:\Users\Asus\anaconda3\envs\ser_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|██████████| 5068/5068 [00:53<00:00, 94.72it/s] 


train saved: (5068, 128, 300) (5068,)


In [8]:
process_split(VAL_CSV, "val")


100%|██████████| 1031/1031 [00:06<00:00, 157.55it/s]


val saved: (1031, 128, 300) (1031,)


In [9]:
process_split(TEST_CSV, "test")


100%|██████████| 1128/1128 [00:07<00:00, 156.99it/s]


test saved: (1128, 128, 300) (1128,)


In [10]:
X_train = np.load(r"D:\SER_Cross\features\train\X.npy")
y_train = np.load(r"D:\SER_Cross\features\train\y.npy")

print(X_train.shape, y_train.shape)


(5068, 128, 300) (5068,)
